In [0]:
-- Create the summerised qdp_banking_supply_chain Data Product in the GOLD Environment from SILVER Layer Data Products
-- N.B.  The GOLD Layer Data Product are not strictly modelled as a Data Vault Schema, because they are generally implemented as a de-normalised Data Mart for reporting consumption

-- 01-1. Create a point in time run_timestamp for all data inserts
DROP TEMPORARY VARIABLE IF EXISTS run_timestamp;

DECLARE run_timestamp TIMESTAMP DEFAULT current_timestamp();

SELECT run_timestamp;


-- 01-2. Create default start date for all data inserts

DROP TEMPORARY VARIABLE IF EXISTS default_start_date;

DECLARE default_start_date STRING;

SET  VAR default_start_date = '01-01-1900';

SELECT default_start_date;


-- 2. Remove all Account Transaction records for the Tennant

DELETE FROM demo_banking_gold.qdp_banking_supply_chain.sat_banking_supply_chain s
WHERE s.banking_supply_chain_id IN (
  SELECT DISTINCT banking_supply_chain_id
  FROM demo_banking_gold.qdp_banking_supply_chain.hub_banking_supply_chain
  WHERE tennant_id = :p_tennant_id)
;
DELETE FROM demo_banking_gold.qdp_banking_supply_chain.sat_currency s
WHERE s.banking_supply_chain_id IN (
  SELECT DISTINCT banking_supply_chain_id
  FROM demo_banking_gold.qdp_banking_supply_chain.hub_banking_supply_chain
  WHERE tennant_id = :p_tennant_id)
;

DELETE FROM demo_banking_gold.qdp_banking_supply_chain.sat_risk_score s
WHERE s.banking_supply_chain_id IN (
  SELECT DISTINCT banking_supply_chain_id
  FROM demo_banking_gold.qdp_banking_supply_chain.hub_banking_supply_chain
  WHERE tennant_id = :p_tennant_id)
;

DELETE FROM demo_banking_gold.qdp_banking_supply_chain.sat_warning_signal s
WHERE s.banking_supply_chain_id IN (
  SELECT DISTINCT banking_supply_chain_id
  FROM demo_banking_gold.qdp_banking_supply_chain.hub_banking_supply_chain
  WHERE tennant_id = :p_tennant_id)
;


-- 3. Select the source data from qdp_aggregated_transaction in Silver
CREATE OR REPLACE TEMP VIEW temp_transactions AS
SELECT 
    h.tennant_id,
    t.tennant_name,
    t.network_name,
    h.bene_account_entity_id,
    h.bene_account_id,
    h.orig_account_entity_id,
    h.orig_account_id,
    s.amount,
    s.datetime,
    s.description,
    s.beneficiary_name,
    s.original_account_name,
    s.period_start,
    s.period_end
  FROM demo_banking_silver.qdp_transactions_all.hub_transaction h
  JOIN demo_banking_silver.qdp_transactions_all.sat_transaction s
    ON h.transaction_id = s.transaction_id
  JOIN qdp_refcode_silver.qdp_reference_codes.tennant t
   ON h.tennant_id = t.tennant_id
  WHERE h.tennant_id = 1
  ;


SELECT * FROM temp_transactions;

   

-- 4. Create a Summary View ready for posting into qdp_baking_supply_chain in Gold

CREATE OR REPLACE TEMP VIEW summary_transactions AS
SELECT
    tennant_id,
    tennant_name,
    network_name,
    bene_account_id,
    bene_account_entity_id,
    orig_account_id,
    orig_account_entity_id,
    count(*) AS total_transactions,
    sum(amount) AS total_value,
    min(amount) AS min_value,
    max(amount) AS max_value,
    avg(amount) AS avg_value,
    min(datetime) AS first_transaction_date,
    max(datetime) AS last_transaction_date

  FROM temp_transactions
--  GROUP BY bene_account_entity_id, orig_account_entity_id
  GROUP BY tennant_id, tennant_name, network_name, bene_account_id, bene_account_entity_id, orig_account_id, orig_account_entity_id
;

SELECT * FROM summary_transactions;



-- 5. Create the  hub_individual table records
INSERT INTO demo_banking_gold.qdp_banking_supply_chain.hub_banking_supply_chain (
    tennant_id,
    tennant_name,
    network_name,
    bene_account_id,
    bene_account_entity_id,
    orig_account_id,
    orig_account_entity_id
    )
SELECT 
    st.tennant_id,
    st.tennant_name,
    st.network_name,
    st.bene_account_id, 
    st.bene_account_entity_id, 
    st.orig_account_id, 
    st.orig_account_entity_id 
  FROM summary_transactions st
;

SELECT * FROM demo_banking_gold.qdp_banking_supply_chain.hub_banking_supply_chain;


-- 6. Create the  sat_banking_supply_chain table records
INSERT INTO demo_banking_gold.qdp_banking_supply_chain.sat_banking_supply_chain
    ( 
    tennant_id,
    tennant_name,
    network_name,
    banking_supply_chain_id,
    bene_account_id,
    orig_account_id,
    load_datetime,
    record_source_id,
	  total_transactions,
	  total_value,
	  max_value,
	  min_value,
    avg_value,
    first_transaction_date,
		last_transaction_date
    )
SELECT
  h.tennant_id AS tennant_id,
  h.tennant_name AS tennant_name,
  h.network_name AS network_name,
  h.banking_supply_chain_id AS banking_supply_chain_id,
  h.bene_account_id AS bene_account_id,
  h.orig_account_id AS orig_account_id,
  run_timestamp AS load_datetime,
  try_cast(0 AS BIGINT) AS record_source_id,
  st.total_transactions,
  st.total_value,
  st.max_value,
  st.min_value,
  st.avg_value,
  st.first_transaction_date,
  st.last_transaction_date
FROM summary_transactions st
JOIN demo_banking_gold.qdp_banking_supply_chain.hub_banking_supply_chain h
  ON ((h.bene_account_id = st.bene_account_id) AND (h.orig_account_id = st.orig_account_id));
  
SELECT * FROM demo_banking_gold.qdp_banking_supply_chain.sat_banking_supply_chain;




-- 6. Create the money supply chain to 3 levels of originators and beneficiaries ()

-- Determine ultimate beneficiaries
CREATE OR REPLACE TEMP VIEW view_third_level_beneficiary AS
SELECT tennant_id, tennant_name, network_name, originator_account_name, beneficiary_account_name, SUM(total_value) as total_value from 
demo_banking_gold.qdp_banking_supply_chain.view_banking_supply_chains
WHERE beneficiary_account_name NOT IN (
  SELECT originator_account_name
  FROM demo_banking_gold.qdp_banking_supply_chain.view_banking_supply_chains
)
GROUP BY tennant_id, tennant_name, network_name, originator_account_name, beneficiary_account_name
;

SELECT * FROM view_third_level_beneficiary;


-- Determine beneficiaries who are originators for ultimate beneficiaries
CREATE OR REPLACE TEMP VIEW view_second_level_beneficiary AS
SELECT tennant_id, tennant_name, network_name, originator_account_name, beneficiary_account_name, SUM(total_value) as total_value from 
demo_banking_gold.qdp_banking_supply_chain.view_banking_supply_chains
WHERE beneficiary_account_name IN (
  SELECT originator_account_name
  FROM view_third_level_beneficiary
)
GROUP BY tennant_id, tennant_name, network_name, originator_account_name, beneficiary_account_name
;

SELECT * FROM view_second_level_beneficiary;


DELETE FROM demo_banking_gold.qdp_banking_supply_chain.fact_cash_flow_3;

INSERT INTO demo_banking_gold.qdp_banking_supply_chain.fact_cash_flow_3 (
  tennant_id, 
  tennant_name,
  network_name,
  stage1, 
  stage2, 
  stage3, 
  value
)
SELECT 
  tennant_id, 
  tennant_name,
  network_name,
  NULL, 
  originator_account_name, 
  beneficiary_account_name, 
  total_value
 FROM view_third_level_beneficiary
;


INSERT INTO demo_banking_gold.qdp_banking_supply_chain.fact_cash_flow_3 (
  tennant_id, 
  tennant_name,
  network_name,
  stage1, 
  stage2, 
  stage3, 
  value
)
SELECT 
  tennant_id, 
  tennant_name,
  network_name,
  originator_account_name, 
  beneficiary_account_name, 
  NULL, 
  total_value
 FROM view_second_level_beneficiary
;


SELECT * FROM demo_banking_gold.qdp_banking_supply_chain.fact_cash_flow_3;

